# 🚦 Segurança Viária — PRF Pará 2024–2025
## Notebook 01 · EDA, Governança e Análise Exploratória

**Autor:** Gilson Machado — Cientista de Dados | Pós-Graduação em Estatística Aplicada à Ciência de Dados  
**Fonte:** PRF/DATATRAN — dados abertos | **Recorte:** Rodovias federais · UF = PA · Jan/2024–Nov/2025

---

| Seção | Conteúdo |
|-------|----------|
| 1 | Bootstrap, carga e validação |
| 2 | Auditoria de qualidade |
| 3 | Engenharia de features |
| 4 | Contexto nacional — PA no ranking de fatalidade |
| 5 | Análise temporal — tendência de acidentes |
| 6 | Análise espacial — mapas de densidade (KDE) |
| 7 | Análise descritiva multivariada |
| 8 | Identificação de trechos críticos |
| 9 | Base exportada para Notebook 02 |

> *Este notebook produz `df_pa_final.parquet` que alimenta o Notebook 02 (modelagem).*


---
## 1. Bootstrap & Carga de Dados

Montagem do Drive e carregamento robusto dos dois arquivos DATATRAN.  
`carregar_datatran` é a **única função canônica** de carga — `decimal=','` correto, validação explícita de colunas e recorte geográfico auditável.


In [1]:
# ── 1.1  DEPENDÊNCIAS ──────────────────────────────────────────────────────────
import os, warnings
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
warnings.filterwarnings("ignore")

# Colab: montar Drive
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as e:
    print("⚠️  Drive indisponível:", e)


Mounted at /content/drive


In [2]:
# ── 1.2  LOCALIZAR ARQUIVOS (independente de subpasta) ─────────────────────────
ALVOS = {"datatran2024.csv", "datatran2025.csv"}
raiz  = "/content/drive/MyDrive"

encontrados = {}
for root, _, files in os.walk(raiz):
    for f in files:
        if f in ALVOS and f not in encontrados:
            encontrados[f] = os.path.join(root, f)

faltando = sorted(ALVOS - set(encontrados.keys()))
if faltando:
    raise FileNotFoundError(
        "❌ Não encontrado no MyDrive: " + ", ".join(faltando)
    )

ARQ_2024, ARQ_2025 = encontrados["datatran2024.csv"], encontrados["datatran2025.csv"]
print("✅ 2024:", ARQ_2024)
print("✅ 2025:", ARQ_2025)


✅ 2024: /content/drive/MyDrive/ppgme_datatran/datatran2024.csv
✅ 2025: /content/drive/MyDrive/ppgme_datatran/datatran2025.csv


In [3]:
# ── 1.3  FUNÇÃO DE CARGA CANÔNICA ──────────────────────────────────────────────
def carregar_datatran(caminho: str, uf: str = "PA") -> pd.DataFrame:
    df = pd.read_csv(
        caminho, sep=";", encoding="latin-1",
        decimal=",", low_memory=False
    )
    essenciais = {"id","data_inversa","uf","municipio","br","km"}
    ausentes = essenciais - set(df.columns)
    if ausentes:
        raise ValueError(f"Colunas ausentes: {sorted(ausentes)}")

    df["data_inversa"] = pd.to_datetime(df["data_inversa"], errors="coerce")
    df = df[df["uf"] == uf].copy()

    if df["uf"].unique().tolist() != [uf]:
        raise ValueError("Recorte UF falhou.")

    cols_int = ["mortos","feridos_graves","feridos_leves",
                "ilesos","feridos","pessoas","veiculos"]
    for c in cols_int:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)
    for col in ["latitude","longitude","km"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


In [4]:
# ── 1.4  CARGA, CONCATENAÇÃO E AUDITORIA RÁPIDA ────────────────────────────────
pa24 = carregar_datatran(ARQ_2024)
pa25 = carregar_datatran(ARQ_2025)
df   = pd.concat([pa24, pa25], ignore_index=True)

print(f"✅ Base PA: {df.shape[0]:,} linhas | {df.shape[1]} colunas")
print(f"   2024: {len(pa24):,} | 2025: {len(pa25):,}")
print(f"   Período: {df['data_inversa'].min().date()} → {df['data_inversa'].max().date()}")
print(f"   Nulos em data_inversa: {df['data_inversa'].isna().sum()}")


✅ Base PA: 1,977 linhas | 30 colunas
   2024: 970 | 2025: 1,007
   Período: 2024-01-01 → 2025-11-30
   Nulos em data_inversa: 0


---
## 2. Auditoria de Qualidade

Checagem sistemática antes de qualquer análise — governança precede modelagem.


In [24]:
# ── 2.1  COBERTURA MENSAL ──────────────────────────────────────────────────────
meses = (df["data_inversa"].dt.to_period("M")
           .value_counts().sort_index()
           .rename("qtd_acidentes"))
print("📆 Registros por mês:")
display(meses.to_frame())


📆 Registros por mês:


,qtd_acidentes
data_inversa,
2024-01,63
2024-02,72
2024-03,79
2024-04,68
2024-05,83
2024-06,70
2024-07,111
2024-08,83
2024-09,84


In [6]:
# ── 2.2  NULOS, DUPLICIDADE E SEVERIDADE ──────────────────────────────────────
print("📉 Nulos (%) — Top 12:")
display((df.isna().mean()*100).round(2).sort_values(ascending=False)
        .head(12).to_frame("nulos (%)"))

chave = ["data_inversa","horario","br","km","municipio",
         "tipo_acidente","causa_acidente"]
dup = df.duplicated(subset=chave).mean()*100
print(f"\n🔎 Duplicidade por chave composta: {dup:.2f}%")

df["is_fatal"] = (df["mortos"] > 0).astype(int)
print(f"\n☠️  Fatalidade: {df['is_fatal'].mean()*100:.2f}%  "
      f"({df['is_fatal'].sum():,} de {len(df):,})")

cols_sev = ["mortos","feridos_graves","feridos_leves","ilesos","veiculos","pessoas"]
print("\n📊 Severidade:")
display(df[cols_sev].describe().T.round(2))


📉 Nulos (%) — Top 12:


,nulos (%)
uop,0.61
delegacia,0.51
dia_semana,0.00
horario,0.00
id,0.00
data_inversa,0.00
km,0.00
municipio,0.00
causa_acidente,0.00
tipo_acidente,0.00



🔎 Duplicidade por chave composta: 0.00%

☠️  Fatalidade: 18.21%  (360 de 1,977)

📊 Severidade:


,count,mean,std,min,25%,50%,75%,max
mortos,1977.0,0.21,0.50,0.0,0.0,0.0,0.0,6.0
feridos_graves,1977.0,0.39,0.79,0.0,0.0,0.0,1.0,13.0
feridos_leves,1977.0,0.69,1.23,0.0,0.0,0.0,1.0,34.0
ilesos,1977.0,1.00,1.18,0.0,0.0,1.0,1.0,15.0
veiculos,1977.0,2.46,1.52,1.0,2.0,2.0,3.0,15.0
pessoas,1977.0,2.82,2.04,1.0,2.0,2.0,3.0,41.0


In [7]:
# ── 2.3  COBERTURA DAS VARIÁVEIS CATEGÓRICAS ──────────────────────────────────
cat_cols = ["causa_acidente","tipo_acidente","classificacao_acidente",
            "fase_dia","sentido_via","condicao_metereologica",
            "tipo_pista","tracado_via","uso_solo","br","municipio"]
print("📋 Variáveis categóricas — cardinalidade e nulos:")
for c in cat_cols:
    print(f"  {c:<32} cat={df[c].nunique():3d}  nulos={df[c].isna().mean()*100:.1f}%")


📋 Variáveis categóricas — cardinalidade e nulos:
  causa_acidente                   cat= 55  nulos=0.0%
  tipo_acidente                    cat= 16  nulos=0.0%
  classificacao_acidente           cat=  3  nulos=0.0%
  fase_dia                         cat=  4  nulos=0.0%
  sentido_via                      cat=  3  nulos=0.0%
  condicao_metereologica           cat=  7  nulos=0.0%
  tipo_pista                       cat=  3  nulos=0.0%
  tracado_via                      cat= 62  nulos=0.0%
  uso_solo                         cat=  2  nulos=0.0%
  br                               cat= 10  nulos=0.0%
  municipio                        cat= 62  nulos=0.0%


---
## 3. Engenharia de Features

Todas as derivações em uma única célula canônica — sem duplicação.


In [8]:
# ── 3.1  FEATURES TEMPORAIS, SEVERIDADE ORDINAL E TRACADO ─────────────────────
# Temporal
df["ano"]     = df["data_inversa"].dt.year
df["mes"]     = df["data_inversa"].dt.month
df["ano_mes"] = df["data_inversa"].dt.to_period("M").astype(str)

h1 = pd.to_datetime(df["horario"].astype(str), format="%H:%M:%S", errors="coerce")
h2 = pd.to_datetime(df["horario"].astype(str), format="%H:%M",    errors="coerce")
hora_final    = h1.fillna(h2)
df["hora_num"] = hora_final.dt.hour.fillna(0).astype(int).clip(0,23)

def faixa_horaria(h):
    if h <  6: return "Madrugada"
    if h < 12: return "Manhã"
    if h < 18: return "Tarde"
    return "Noite"
df["faixa_horaria"] = df["hora_num"].apply(faixa_horaria)

dias_map = {
    "segunda-feira":1,"segunda":1,"terça-feira":2,"terca-feira":2,
    "terça":2,"terca":2,"quarta-feira":3,"quarta":3,"quinta-feira":4,
    "quinta":4,"sexta-feira":5,"sexta":5,"sábado":6,"sabado":6,"domingo":7
}
df["dia_semana_num"] = (df["dia_semana"].astype(str)
                         .str.strip().str.lower().map(dias_map))

# Severidade ordinal (4 níveis)
df["severidade"] = 0
df.loc[df["feridos_leves"] > 0, "severidade"] = 1
df.loc[df["feridos_graves"] > 0, "severidade"] = 2
df.loc[df["mortos"] > 0, "severidade"] = 3

SEV_LABEL = {0:"Sem vítimas", 1:"Feridos leves",
             2:"Feridos graves", 3:"Fatal"}
df["severidade_label"] = df["severidade"].map(SEV_LABEL)

# Traçado simplificado
def simplifica_tracado(t):
    t = str(t).lower()
    if "curva"    in t: return "Curva"
    if "interseção" in t or "intersecao" in t: return "Interseção"
    if "declive"  in t or "aclive" in t: return "Declive/Aclive"
    if "ponte"    in t or "viaduto" in t or "túnel" in t: return "Obra de Arte"
    if "reta"     in t: return "Reta"
    return "Outro"
df["tracado_simples"] = df["tracado_via"].apply(simplifica_tracado)

# Km numérico e segmento de 10km
df["km_num"] = pd.to_numeric(df["km"], errors="coerce")
df["seg10"]  = (df["km_num"] // 10 * 10).astype("Int64")

print("✅ Engenharia de features concluída")
print(f"   Nulos hora_final (%): {hora_final.isna().mean()*100:.2f}")
print(f"   Nulos dia_semana_num (%): {df['dia_semana_num'].isna().mean()*100:.2f}")
print()
print("Distribuição de severidade:")
for k,v in df["severidade"].value_counts().sort_index().items():
    print(f"  {k} — {SEV_LABEL[k]}: {v:,} ({v/len(df)*100:.1f}%)")


✅ Engenharia de features concluída
   Nulos hora_final (%): 0.00
   Nulos dia_semana_num (%): 0.00

Distribuição de severidade:
  0 — Sem vítimas: 384 (19.4%)
  1 — Feridos leves: 742 (37.5%)
  2 — Feridos graves: 491 (24.8%)
  3 — Fatal: 360 (18.2%)


---
## 4. Contexto Nacional — PA no Ranking de Fatalidade

O Pará ocupa o **2° lugar nacional** em taxa de fatalidade em rodovias federais (2024–2025),  
com 18,2% — **2,5× a média nacional** de 7,2%. Este é o dado que justifica e motiva todo o estudo.


In [9]:
# ── 4.1  RANKING NACIONAL DE FATALIDADE ───────────────────────────────────────
df24_nac = pd.read_csv(ARQ_2024, sep=";", encoding="latin-1",
                       decimal=",", low_memory=False)
df25_nac = pd.read_csv(ARQ_2025, sep=";", encoding="latin-1",
                       decimal=",", low_memory=False)
df_nac = pd.concat([df24_nac, df25_nac], ignore_index=True)
df_nac["mortos"] = pd.to_numeric(df_nac["mortos"], errors="coerce").fillna(0)
df_nac["is_fatal"] = (df_nac["mortos"] > 0).astype(int)

by_uf = (df_nac.groupby("uf")
               .agg(n=("id","count"), fatais=("is_fatal","sum"),
                    taxa=("is_fatal","mean"))
               .assign(taxa_pct=lambda x: (x["taxa"]*100).round(2))
               .query("n >= 200")
               .sort_values("taxa_pct", ascending=True))

cores = ["#E63946" if uf == "PA" else "#B0BEC5" for uf in by_uf.index]

fig = go.Figure(go.Bar(
    x=by_uf["taxa_pct"], y=by_uf.index,
    orientation="h",
    marker_color=cores,
    text=by_uf["taxa_pct"].apply(lambda x: f"{x:.1f}%"),
    textposition="outside",
    hovertemplate="%{y}: %{x:.2f}%<extra></extra>"
))
media_nac = df_nac["is_fatal"].mean()*100
fig.add_vline(x=media_nac, line_dash="dash", line_color="#546E7A",
              annotation_text=f"Média BR: {media_nac:.1f}%",
              annotation_position="top right")
fig.update_layout(
    title=dict(text="Taxa de Fatalidade por UF — Rodovias Federais 2024–2025<br>"
               "<sup>PA em destaque · apenas UFs com ≥200 acidentes</sup>",
               x=0.02),
    xaxis_title="Taxa de fatalidade (%)",
    yaxis_title="",
    height=680, width=820,
    plot_bgcolor="white", paper_bgcolor="white",
    margin=dict(l=60,r=80,t=80,b=50),
    xaxis=dict(showgrid=True, gridcolor="lightgray")
)
fig.show()
pa_rank = len(by_uf) - by_uf.index.tolist().index("PA")
print(f"✅ PA ocupa o {pa_rank}° lugar (mais letal) entre {len(by_uf)} UFs com n≥200")
print(f"   Taxa PA: {by_uf.loc['PA','taxa_pct']:.2f}%  |  Média nacional: {media_nac:.2f}%")


✅ PA ocupa o 2° lugar (mais letal) entre 27 UFs com n≥200
   Taxa PA: 18.21%  |  Média nacional: 7.17%


---
## 5. Análise Temporal — Tendência de Acidentes

**GLM Poisson** com controle sazonal `C(mes)` e termo de tendência `tempo`.  
Pipeline: Poisson base → diagnóstico sobredispersão (Pearson χ²/df) → **Poisson HC0** (principal) → NB2 (sensibilidade).


In [10]:
# ── 5.1  SÉRIE MENSAL ──────────────────────────────────────────────────────────
df_mensal = (df.groupby("ano_mes", as_index=False)
               .agg(qtd_acidentes=("id","count"),
                    n_fatais=("is_fatal","sum")))
df_mensal["ano_mes_dt"] = pd.to_datetime(
    df_mensal["ano_mes"] + "-01", errors="coerce")
df_mensal = (df_mensal.dropna(subset=["ano_mes_dt"])
                      .sort_values("ano_mes_dt")
                      .reset_index(drop=True))
df_mensal["taxa_fatal"] = (df_mensal["n_fatais"] /
                            df_mensal["qtd_acidentes"] * 100).round(2)
media = df_mensal["qtd_acidentes"].mean()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=("Nº de Acidentes por Mês",
                                    "Taxa de Fatalidade Mensal (%)"),
                    vertical_spacing=0.10)

fig.add_trace(go.Scatter(
    x=df_mensal["ano_mes_dt"], y=df_mensal["qtd_acidentes"],
    mode="lines+markers", name="Acidentes",
    line=dict(color="#2D6A9F", width=2), marker=dict(size=6)), row=1, col=1)
fig.add_trace(go.Scatter(
    x=df_mensal["ano_mes_dt"], y=[media]*len(df_mensal),
    mode="lines", name=f"Média: {media:.0f}",
    line=dict(dash="dash", color="#E74C3C", width=1.5)), row=1, col=1)
fig.add_trace(go.Bar(
    x=df_mensal["ano_mes_dt"], y=df_mensal["taxa_fatal"],
    name="Taxa fatal (%)", marker_color="#E67E22", opacity=0.7), row=2, col=1)

fig.update_layout(
    title=dict(text="📈 Evolução Mensal — Acidentes e Fatalidade | Pará 2024–2025<br>"
               "<sup>Linha vermelha = média descritiva (tendência inferida pelo GLM na célula 5.2)</sup>",
               x=0.02),
    height=560, width=1000,
    plot_bgcolor="white", paper_bgcolor="white",
    margin=dict(l=60,r=30,t=80,b=50))
fig.update_xaxes(showgrid=True, gridcolor="lightgray", tickformat="%b %Y")
fig.update_yaxes(showgrid=True, gridcolor="lightgray", rangemode="tozero")
fig.show()


In [11]:
# ── 5.2  GLM POISSON — DIAGNÓSTICO + HC0 + NB2 ────────────────────────────────
import statsmodels.api as sm
import statsmodels.formula.api as smf
import patsy

df_mensal["tempo"] = range(len(df_mensal))
df_mensal["mes"]   = df_mensal["ano_mes_dt"].dt.month.astype(int)

# Poisson base
mod_pois = smf.glm("qtd_acidentes ~ tempo + C(mes)",
                   data=df_mensal, family=sm.families.Poisson()).fit()
pearson_ratio = mod_pois.pearson_chi2 / mod_pois.df_resid

print("="*60)
print("🔎 DIAGNÓSTICO DE SOBREDISPERSÃO")
print("="*60)
print(f"Pearson χ²/df = {pearson_ratio:.3f}")
print("⚠️  Sobredispersão → adotar HC0" if pearson_ratio > 1.2
      else "✅ Sem sobredispersão relevante")

# Poisson robusto HC0 — especificação principal
mod_rob = smf.glm("qtd_acidentes ~ tempo + C(mes)",
                  data=df_mensal, family=sm.families.Poisson()
                  ).fit(cov_type="HC0")
coef = float(mod_rob.params.get("tempo", np.nan))
p_v  = float(mod_rob.pvalues.get("tempo", np.nan))
pct  = (np.exp(coef)-1)*100

print(f"\n📌 POISSON HC0 — TENDÊNCIA TEMPORAL")
print(f"   Coef(tempo): {coef:.4f}  →  {pct:+.2f}%/mês")
print(f"   P-valor: {p_v:.4f}  → {'✅ Significativo' if p_v<0.05 else '❌ Não significativo'}")

# NB2 — sensibilidade
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    y_nb, X_nb = patsy.dmatrices(
        "qtd_acidentes ~ tempo + C(mes)", df_mensal, return_type="dataframe")
    mod_nb2 = sm.NegativeBinomial(y_nb, X_nb).fit(disp=False)

alpha = float(mod_nb2.params.iloc[-1])
p_nb2 = float(mod_nb2.pvalues.get("tempo", np.nan))
print(f"\n📌 NB2 — SENSIBILIDADE")
print(f"   α = {alpha:.4f} {'(→ colapsa para Poisson)' if alpha < 0.05 else ''}")
print(f"   P-valor(tempo): {p_nb2:.4f}")
print("   Conclusão: resultados consistentes com Poisson HC0.")

display(mod_rob.summary())


🔎 DIAGNÓSTICO DE SOBREDISPERSÃO
Pearson χ²/df = 1.592
⚠️  Sobredispersão → adotar HC0

📌 POISSON HC0 — TENDÊNCIA TEMPORAL
   Coef(tempo): 0.0089  →  +0.89%/mês
   P-valor: 0.0081  → ✅ Significativo

📌 NB2 — SENSIBILIDADE
   α = 0.0000 (→ colapsa para Poisson)
   P-valor(tempo): 0.0197
   Conclusão: resultados consistentes com Poisson HC0.


<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:          qtd_acidentes   No. Observations:                   23
Model:                            GLM   Df Residuals:                       10
Model Family:                 Poisson   Df Model:                           12
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -80.158
Date:                Tue, 19 May 2026   Deviance:                       15.938
Time:                        18:52:11   Pearson chi2:                     15.9
No. Iterations:                     4   Pseudo R-squ. (CS):             0.8787
Covariance Type:                  HC0                                         
================================================================================
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept        4.1118      0.030    136.454      0.000       4.053       4.171
C(mes)[T.2]      0.1080      0.039      2.749      0.006       0.031       0.185
C(mes)[T.3]      0.1528      0.065      2.356      0.018       0.026       0.280
C(mes)[T.4]      0.1887      0.072      2.605      0.009       0.047       0.331
C(mes)[T.5]      0.2463      0.030      8.101      0.000       0.187       0.306
C(mes)[T.6]      0.2256      0.088      2.558      0.011       0.053       0.398
C(mes)[T.7]      0.4195      0.094      4.456      0.000       0.235       0.604
C(mes)[T.8]      0.3354      0.066      5.049      0.000       0.205       0.466
C(mes)[T.9]      0.3265      0.061      5.313      0.000       0.206       0.447
C(mes)[T.10]     0.3484      0.103      3.398      0.001       0.147       0.549
C(mes)[T.11]     0.3293      0.069      4.763      0.000       0.194       0.465
C(mes)[T.12]    -0.0354      0.026     -1.338      0.181      -0.087       0.016
tempo            0.0089      0.003      2.646      0.008       0.002       0.015
================================================================================
"""

In [12]:
# ── 5.3  OBSERVADO vs AJUSTADO ─────────────────────────────────────────────────
df_plot = df_mensal.copy()
df_plot["ajustado"] = mod_rob.predict(df_plot)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_plot["ano_mes_dt"], y=df_plot["qtd_acidentes"],
    mode="lines+markers", name="Observado",
    line=dict(color="#2D6A9F", width=2), marker=dict(size=6)))
fig.add_trace(go.Scatter(
    x=df_plot["ano_mes_dt"], y=df_plot["ajustado"],
    mode="lines", name="Ajustado (Poisson HC0)",
    line=dict(color="#E67E22", width=2.5, dash="dash")))
fig.update_layout(
    title=dict(text="Acidentes Mensais: Observado vs Ajustado — Poisson HC0 | Pará 2024–2025",
               x=0.02),
    xaxis_title="Mês", yaxis_title="Nº de acidentes",
    height=420, width=1000,
    plot_bgcolor="white", paper_bgcolor="white",
    margin=dict(l=60,r=30,t=60,b=50))
fig.update_xaxes(showgrid=True, gridcolor="lightgray", tickformat="%b %Y")
fig.update_yaxes(showgrid=True, gridcolor="lightgray", rangemode="tozero")
fig.show()


### 📋 Interpretação — Tendência Temporal

- **Pearson χ²/df ≈ 1,59** → sobredispersão moderada, justifica erros robustos HC0  
- **Crescimento estimado: +0,89%/mês** (p = 0,008) — mantido após NB2 (α → 0)  
- **Limitação:** série de 23 meses limita separação de tendência vs ciclos de médio prazo

> *AIC/BIC mantidos no sumário para comparabilidade entre especificações. NB2 com ConvergenceWarning é esperado quando α → 0 (não-convergência confirma colapso para Poisson).*


---
## 6. Análise Espacial — Mapas de Densidade (KDE)

Três mapas complementares para leitura espacial completa.  
O traçado das rodovias federais do Pará emerge organicamente da densidade dos acidentes — sem necessidade de shapefile externo.

> *KDE é análise exploratória/descritiva. Não controla exposição ao risco (volume de tráfego). Hotspot com teste estatístico está no Notebook 02.*


In [13]:
# ── 6.1  MAPA 1: INTENSIDADE DE MORTES (z = mortos) ──────────────────────────
df_map = df[(df["data_inversa"].dt.year >= 2024) &
            (df["data_inversa"].dt.year <= 2025)].copy()

df_mortos = df_map.dropna(subset=["latitude","longitude"]).query("mortos > 0").copy()

fig = px.density_map(
    df_mortos, lat="latitude", lon="longitude", z="mortos",
    radius=12,
    center=dict(lat=-4.5, lon=-52.0), zoom=5,
    map_style="open-street-map",
    title="🗺️ Intensidade de Mortes em Acidentes — Kernel Density | Pará 2024–2025"
)
fig.update_layout(width=900, height=650,
                  margin={"r":0,"t":50,"l":0,"b":0})
fig.show()
print(f"✅ Acidentes com mortes: {len(df_mortos):,} | Total de mortos: {int(df_mortos['mortos'].sum()):,}")


✅ Acidentes com mortes: 360 | Total de mortos: 418


### 📋 Interpretação — Mapa de Mortes

O traçado das rodovias federais do PA emerge da concentração de óbitos: a **BR-316**, **BR-230** (Transamazônica) e **BR-163** são claramente identificáveis.  
Manchas mais quentes (amarelo/branco) indicam trechos com múltiplas mortes por evento — não apenas frequência de acidentes.


In [14]:
# ── 6.2  MAPA 2: OCORRÊNCIA DE ACIDENTES FATAIS (z = is_fatal) ───────────────
df_fatal = df_map.dropna(subset=["latitude","longitude"]).query("is_fatal == 1").copy()

fig = px.density_map(
    df_fatal, lat="latitude", lon="longitude", z="is_fatal",
    radius=12,
    center=dict(lat=-4.5, lon=-52.0), zoom=5,
    map_style="open-street-map",
    title="🗺️ Densidade de Acidentes Fatais — Kernel Density | Pará 2024–2025"
)
fig.update_layout(width=900, height=650,
                  margin={"r":0,"t":50,"l":0,"b":0})
fig.show()
print(f"✅ Acidentes fatais plotados: {len(df_fatal):,}")


✅ Acidentes fatais plotados: 360


### 📋 Interpretação — Mapa de Acidentes Fatais

Diferente do mapa de mortes (que pondera severidade), este captura **frequência de eventos fatais** — peso 1 por evento, independente de quantas vítimas.  
Permite distinguir trechos com **alta letalidade por evento** (mapa anterior) de trechos com **alta repetição de fatalidades** (este mapa).


In [15]:
# ── 6.3  MAPA 3: TODOS OS ACIDENTES COM ESCALA DE SEVERIDADE ─────────────────
# Mapa complementar: visualiza a distribuição espacial por nível de severidade
cores_sev = {0:"#95A5A6", 1:"#F39C12", 2:"#E67E22", 3:"#E74C3C"}

df_geo = df_map.dropna(subset=["latitude","longitude"]).copy()
df_geo["cor"] = df_geo["severidade"].map(cores_sev)
df_geo["tam"] = df_geo["severidade"].map({0:4, 1:5, 2:6, 3:8})

fig = px.scatter_map(
    df_geo,
    lat="latitude", lon="longitude",
    color="severidade_label",
    color_discrete_map={
        "Sem vítimas":"#95A5A6","Feridos leves":"#F39C12",
        "Feridos graves":"#E67E22","Fatal":"#E74C3C"},
    size="tam",
    size_max=8,
    opacity=0.6,
    center=dict(lat=-4.5, lon=-52.0), zoom=5,
    map_style="open-street-map",
    title="🗺️ Distribuição Espacial por Severidade | Pará 2024–2025",
    hover_data=["municipio","br","causa_acidente","mortos"]
)
fig.update_layout(width=900, height=650,
                  margin={"r":0,"t":50,"l":0,"b":0})
fig.show()


### 📋 Interpretação — Mapa de Severidade

Permite sobrepor visualmente os quatro níveis de desfecho no espaço geográfico.  
Pontos vermelhos (fatal) concentrados em trechos específicos evidenciam os hotspots que serão testados estatisticamente no Notebook 02.


---
## 7. Análise Descritiva Multivariada

Distribuições das variáveis-chave com leitura de fatalidade por categoria.  
Estas distribuições informam a seleção de features e interpretação dos modelos no Notebook 02.


In [16]:
# ── 7.1  FATALIDADE POR VARIÁVEL CATEGÓRICA ───────────────────────────────────
def plot_fatalidade_cat(col, titulo, top_n=12, min_n=20):
    tab = (df.groupby(col)
             .agg(n=("id","count"), fatais=("is_fatal","sum"))
             .assign(pct=lambda x: x["fatais"]/x["n"]*100)
             .query(f"n >= {min_n}")
             .sort_values("pct", ascending=True)
             .tail(top_n))

    fig = go.Figure()
    fig.add_trace(go.Bar(
        y=tab.index, x=tab["pct"],
        orientation="h",
        marker_color="#E74C3C", opacity=0.8,
        text=tab["pct"].apply(lambda x: f"{x:.1f}%  (n={tab.loc[tab.index==tab.index[tab['pct'].tolist().index(x)], 'n'].values[0]:,})"),
        textposition="outside"
    ))
    fig.add_vline(x=df["is_fatal"].mean()*100, line_dash="dash",
                  line_color="#2D6A9F",
                  annotation_text=f"Média PA: {df['is_fatal'].mean()*100:.1f}%")
    fig.update_layout(
        title=dict(text=titulo, x=0.02),
        xaxis_title="Taxa de fatalidade (%)",
        height=max(350, top_n*38), width=820,
        plot_bgcolor="white", paper_bgcolor="white",
        margin=dict(l=200,r=120,t=60,b=40),
        showlegend=False)
    fig.update_xaxes(showgrid=True, gridcolor="lightgray", range=[0, None])
    fig.show()

plot_fatalidade_cat("causa_acidente",
    "Taxa de Fatalidade por Causa do Acidente (n≥20) | Pará 2024–2025")


In [17]:
plot_fatalidade_cat("tipo_acidente",
    "Taxa de Fatalidade por Tipo de Acidente (n≥20) | Pará 2024–2025")


In [18]:
# ── 7.2  FATALIDADE POR USO_SOLO, TRAÇADO E TIPO DE PISTA ────────────────────
fig = make_subplots(rows=1, cols=3,
                    subplot_titles=("Uso do Solo","Traçado da Via","Tipo de Pista"))

for col_idx, (col, label) in enumerate(
        [("uso_solo","Zona"), ("tracado_simples","Traçado"),
         ("tipo_pista","Pista")], 1):
    tab = (df.groupby(col)
             .agg(n=("id","count"), pct=("is_fatal","mean"))
             .assign(pct=lambda x: x["pct"]*100)
             .sort_values("pct", ascending=False))
    fig.add_trace(go.Bar(
        x=tab.index, y=tab["pct"],
        marker_color=["#E74C3C" if v == tab["pct"].max()
                      else "#85B7EB" for v in tab["pct"]],
        showlegend=False,
        text=tab["pct"].apply(lambda x: f"{x:.1f}%"),
        textposition="outside"
    ), row=1, col=col_idx)

fig.update_layout(
    title=dict(text="Taxa de Fatalidade por Características da Via | Pará 2024–2025",
               x=0.02),
    height=420, width=1000,
    plot_bgcolor="white", paper_bgcolor="white",
    margin=dict(l=40,r=40,t=80,b=60))
fig.update_yaxes(showgrid=True, gridcolor="lightgray",
                 title_text="Taxa fatal (%)", rangemode="tozero")
fig.show()
print("Uso do Solo: Não = rural (21,3% fatal)  |  Sim = urbano (12,3% fatal)")


Uso do Solo: Não = rural (21,3% fatal)  |  Sim = urbano (12,3% fatal)


In [19]:
# ── 7.3  FATALIDADE POR HORA E FAIXA HORÁRIA × USO_SOLO ─────────────────────
tab_hora = (df.groupby(["hora_num","uso_solo"])["is_fatal"]
              .agg(["sum","count","mean"])
              .reset_index()
              .rename(columns={"sum":"fatais","count":"total","mean":"taxa"}))
tab_hora["taxa"] *= 100

fig = go.Figure()
for uso, cor in [("Não","#E74C3C"), ("Sim","#2D6A9F")]:
    sub = tab_hora[tab_hora["uso_solo"]==uso]
    label = "Rural" if uso=="Não" else "Urbano"
    fig.add_trace(go.Scatter(
        x=sub["hora_num"], y=sub["taxa"],
        mode="lines+markers", name=label,
        line=dict(color=cor, width=2), marker=dict(size=5)))

fig.update_layout(
    title=dict(
        text="Taxa de Fatalidade por Hora do Dia × Uso do Solo | Pará 2024–2025<br>"
             "<sup>Rural = uso_solo:Não | Urbano = uso_solo:Sim</sup>",
        x=0.02),
    xaxis=dict(title="Hora do dia", dtick=2,
               ticktext=["00h","02h","04h","06h","08h","10h","12h",
                         "14h","16h","18h","20h","22h"],
               tickvals=list(range(0,24,2))),
    yaxis_title="Taxa de fatalidade (%)",
    height=400, width=900,
    plot_bgcolor="white", paper_bgcolor="white",
    margin=dict(l=60,r=30,t=80,b=50))
fig.update_xaxes(showgrid=True, gridcolor="lightgray")
fig.update_yaxes(showgrid=True, gridcolor="lightgray", rangemode="tozero")
fig.show()


In [20]:
# ── 7.4  DISTRIBUIÇÃO DE SEVERIDADE POR RODOVIA ───────────────────────────────
brs_ord = (df.groupby("br")["is_fatal"].mean()
             .sort_values(ascending=False)
             .index.tolist())

tab_sev_br = (df[df["br"].isin(brs_ord)]
               .groupby(["br","severidade_label"])
               .size().reset_index(name="n"))
tab_sev_br["pct"] = (tab_sev_br.groupby("br")["n"]
                      .transform(lambda x: x/x.sum()*100))

fig = px.bar(
    tab_sev_br, x="br", y="pct", color="severidade_label",
    category_orders={
        "br": brs_ord,
        "severidade_label": ["Fatal","Feridos graves","Feridos leves","Sem vítimas"]
    },
    color_discrete_map={
        "Fatal":"#E74C3C","Feridos graves":"#E67E22",
        "Feridos leves":"#F1C40F","Sem vítimas":"#BDC3C7"},
    title="Distribuição de Severidade por Rodovia | Pará 2024–2025",
    labels={"pct":"Proporção (%)","br":"Rodovia","severidade_label":"Severidade"},
    barmode="stack"
)
fig.update_layout(
    height=420, width=820,
    plot_bgcolor="white", paper_bgcolor="white",
    margin=dict(l=60,r=30,t=70,b=50))
fig.update_yaxes(showgrid=True, gridcolor="lightgray")
fig.show()


---
## 8. Identificação de Trechos Críticos

Análise por segmentos de 10km (BR + intervalo de km).  
Esta seção produz a tabela de trechos que alimentará o **hotspot inferencial** do Notebook 02.


In [21]:
# ── 8.1  TABELA DE TRECHOS CRÍTICOS ───────────────────────────────────────────
trechos = (df.groupby(["br","seg10"])
             .agg(
                 n_acidentes=("id","count"),
                 n_fatais=("is_fatal","sum"),
                 total_mortos=("mortos","sum"),
                 lat_media=("latitude","mean"),
                 lon_media=("longitude","mean")
             ).reset_index())
trechos["taxa_fatal"] = (trechos["n_fatais"] /
                          trechos["n_acidentes"] * 100).round(1)
trechos["mortos_por_acidente"] = (trechos["total_mortos"] /
                                   trechos["n_acidentes"]).round(3)

# Filtro: pelo menos 5 acidentes
trechos_filtrados = (trechos[trechos["n_acidentes"] >= 5]
                      .sort_values("n_fatais", ascending=False))

print(f"✅ Trechos com n≥5: {len(trechos_filtrados)}")
print(f"   Taxa fatal média: {trechos_filtrados['taxa_fatal'].mean():.1f}%")
print()
print("🔴 Top 15 trechos por número de acidentes fatais:")
display(trechos_filtrados.head(15)[
    ["br","seg10","n_acidentes","n_fatais",
     "total_mortos","taxa_fatal","mortos_por_acidente"]])


✅ Trechos com n≥5: 117
   Taxa fatal média: 20.5%

🔴 Top 15 trechos por número de acidentes fatais:


,br,seg10,n_acidentes,n_fatais,total_mortos,taxa_fatal,mortos_por_acidente
254,316,60,85,12,14,14.1,0.165
178,230,120,103,9,9,8.7,0.087
250,316,20,53,8,8,15.1,0.151
251,316,30,44,8,10,18.2,0.227
256,316,80,21,8,11,38.1,0.524
255,316,70,26,7,8,26.9,0.308
177,230,110,114,6,6,5.3,0.053
32,10,320,19,6,6,31.6,0.316
31,10,310,12,6,8,50.0,0.667
144,163,980,36,5,5,13.9,0.139


In [22]:
# ── 8.2  MAPA DE TRECHOS CRÍTICOS ─────────────────────────────────────────────
top_trechos = trechos_filtrados.dropna(subset=["lat_media","lon_media"]).head(30)
top_trechos["label"] = (top_trechos["br"].astype(str) + " km " +
                         top_trechos["seg10"].astype(str) + "–" +
                         (top_trechos["seg10"]+10).astype(str))

fig = px.scatter_map(
    top_trechos,
    lat="lat_media", lon="lon_media",
    size="n_fatais",
    color="taxa_fatal",
    color_continuous_scale="YlOrRd",
    size_max=20,
    hover_name="label",
    hover_data={"n_acidentes":True,"n_fatais":True,
                "taxa_fatal":True,"total_mortos":True,
                "lat_media":False,"lon_media":False},
    center=dict(lat=-4.5, lon=-52.0), zoom=5,
    map_style="open-street-map",
    title="🔴 Top 30 Trechos Críticos (n≥5 acidentes) | Pará 2024–2025<br>"
          "<sup>Tamanho = nº de acidentes fatais · Cor = taxa de fatalidade (%)</sup>"
)
fig.update_layout(width=900, height=650,
                  margin={"r":0,"t":60,"l":0,"b":0})
fig.show()


In [27]:
# ── 8.3  GRÁFICO DE BOLHAS — TRECHOS CRÍTICOS ─────────────────────────────────
import plotly.graph_objects as go

# Preparar dados
t = trechos_filtrados.copy()
t["trecho"] = "BR-" + t["br"].astype(str) + " km " + \
               t["seg10"].astype(str) + "–" + (t["seg10"]+10).astype(str)

# Filtrar top 15 por fatais para não poluir
t15 = t.sort_values("n_fatais", ascending=False).head(15).copy()
t15 = t15.sort_values("taxa_fatal", ascending=True)  # ordenar por taxa no eixo Y

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=t15["taxa_fatal"],
    y=t15["trecho"],
    mode="markers+text",
    marker=dict(
        size=t15["total_mortos"] * 6,
        color=t15["n_acidentes"],
        colorscale="Blues",
        showscale=True,
        colorbar=dict(title="Nº de<br>acidentes"),
        line=dict(color="#333333", width=0.8),
        opacity=0.85
    ),
    text=t15["total_mortos"].apply(lambda x: f"{x} mortos"),
    textposition="middle right",
    textfont=dict(size=10, color="#333333"),
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Taxa de fatalidade: %{x:.1f}%<br>"
        "Acidentes fatais: %{customdata[0]}<br>"
        "Total de mortos: %{customdata[1]}<br>"
        "Total de acidentes: %{customdata[2]}<extra></extra>"
    ),
    customdata=t15[["n_fatais","total_mortos","n_acidentes"]].values
))

# Linha de referência: taxa estadual
taxa_pa = 18.21
fig.add_vline(
    x=taxa_pa,
    line_dash="dash",
    line_color="#E74C3C",
    line_width=1.5,
    annotation_text=f"Média PA: {taxa_pa}%",
    annotation_position="top right",
    annotation_font=dict(color="#E74C3C", size=11)
)

fig.update_layout(
    title=dict(
        text="Trechos Críticos — Taxa de Fatalidade e Severidade | Pará 2024–2025<br>"
             "<sup>Tamanho da bolha = total de mortos · Cor = nº de acidentes · "
             "Linha vermelha = média estadual (18,2%)</sup>",
        x=0.02
    ),
    xaxis=dict(
        title="Taxa de fatalidade (%)",
        showgrid=True,
        gridcolor="lightgray",
        range=[0, max(t15["taxa_fatal"]) * 1.25]
    ),
    yaxis=dict(
        title="",
        showgrid=False
    ),
    height=520,
    width=920,
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(l=180, r=120, t=90, b=60)
)

fig.show()

print()
print("Leitura do gráfico:")
print("  Bolhas à DIREITA da linha = trechos acima da média estadual de fatalidade")
print("  Bolhas MAIORES = mais mortos por trecho (severidade absoluta)")
print("  Bolhas mais ESCURAS = mais acidentes no período (frequência)")
print()
alto_risco = t15[t15["taxa_fatal"] > taxa_pa]
print(f"  Trechos acima da média PA: {len(alto_risco)} de {len(t15)}")
print(f"  Trecho mais letal: {t15.iloc[-1]['trecho']} "
      f"({t15.iloc[-1]['taxa_fatal']:.1f}% | {t15.iloc[-1]['total_mortos']} mortos)")


Leitura do gráfico:
  Bolhas à DIREITA da linha = trechos acima da média estadual de fatalidade
  Bolhas MAIORES = mais mortos por trecho (severidade absoluta)
  Bolhas mais ESCURAS = mais acidentes no período (frequência)

  Trechos acima da média PA: 8 de 15
  Trecho mais letal: BR-10 km 310–320 (50.0% | 8 mortos)


---
## 9. Exportar Base para Notebook 02

Salva `df_pa_final.parquet` com todas as features derivadas — alimenta diretamente o Notebook 02 (Modelagem).


In [28]:
# ── 9.1  EXPORTAR PARQUET ──────────────────────────────────────────────────────
from google.colab import drive
output_path = "/content/drive/MyDrive/ppgme_datatran/df_pa_final.parquet"

# Garantir que dia_semana_num é nullable integer (aceita NaN sem virar float)
df["dia_semana_num"] = df["dia_semana_num"].astype("Int64")  # I maiúsculo

df.to_parquet(output_path, index=False)
print(f"✅ Base exportada: {output_path}")
print(f"   Linhas: {len(df):,} | Colunas: {df.shape[1]}")
print()
print("Colunas disponíveis no Notebook 02:")
for c in sorted(df.columns):
    print(f"  {c}")


✅ Base exportada: /content/drive/MyDrive/ppgme_datatran/df_pa_final.parquet
   Linhas: 1,977 | Colunas: 42

Colunas disponíveis no Notebook 02:
  ano
  ano_mes
  br
  causa_acidente
  classificacao_acidente
  condicao_metereologica
  data_inversa
  delegacia
  dia_semana
  dia_semana_num
  faixa_horaria
  fase_dia
  feridos
  feridos_graves
  feridos_leves
  hora_num
  horario
  id
  ignorados
  ilesos
  is_fatal
  km
  km_num
  latitude
  longitude
  mes
  mortos
  municipio
  pessoas
  regional
  seg10
  sentido_via
  severidade
  severidade_label
  tipo_acidente
  tipo_pista
  tracado_simples
  tracado_via
  uf
  uop
  uso_solo
  veiculos


---
## Resumo do Notebook 01

| Eixo | Resultado |
|------|-----------|
| Base | 1.977 acidentes · 30 colunas · 0% nulos nas vars principais |
| Contexto | PA = **2° lugar nacional** em fatalidade (18,2% vs 7,2% BR) |
| Temporal | Tendência +0,89%/mês (p=0,008) · Poisson HC0 |
| Espacial | 3 mapas KDE · traçado das BRs emerge organicamente |
| Trechos | BR-10 km 310–319: 50% fatal · BR-316 km 80–89: 38% |
| Exportado | `df_pa_final.parquet` → Notebook 02 |

> *Notebook 02: Regressão Ordinal + Modelo Multinível + Hotspot Inferencial + Classificador Calibrado*
